# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto:** Uso de redes sociales y salud mental autopercibida en estudiantes de enseñanza media (YRBS 2023)
**Asignatura:** MCDI500 · Programación para la Ciencia de Datos · **Grupo 8**
**Docente:** Dr. Omar Salinas Silva
**Integrantes:** Matías Manríquez Ortiz · Daniel Pérez Ramirez · Abigail Robles Chávez · Roberto Sánchez Saldivia
**Repositorio:** https://github.com/MatiasManriquezO/proyecto-grupo8-mcdi500

---

### ¿Qué produce la Fase 1?

La Fase 2 recibe el archivo del YRBS 2023 y lo deja listo para analizar. Antes de eso hay que
decidir **qué problema se resuelve, con qué datos y sobre qué entorno**. Ese es el trabajo de
esta fase, y el flujo del cuaderno es:

**Definir → Configurar → Verificar → Estructurar → Documentar → Versionar**

Cada paso se implementa como una **función** reutilizable y termina en una **comprobación con
evidencia** (una salida impresa o un `assert`).

> **Lo que esta fase NO hace.** No limpia, no imputa, no recodifica ni escala. El archivo
> original solo se abre en la sección 5.4 para contrastar su estructura con la ficha declarada.
> Toda transformación ocurre en `F2/notebooks/S1_F2_Preprocesamiento.ipynb`, con las funciones
> de `src/procesamiento.py`.

## Introducción

El proyecto trabaja con el *2023 National Youth Risk Behavior Survey* (YRBS) del CDC, una
encuesta real aplicada a estudiantes de enseñanza media de Estados Unidos. La unidad de análisis
es **un estudiante encuestado**. El archivo original tiene **20.103 registros × 117 variables**;
de ellas, el proyecto selecciona **11 columnas** (siete variables de análisis, un identificador y
las tres del diseño muestral). La reducción es de columnas, no de filas: el subconjunto de trabajo
conserva los 20.103 registros.

El proyecto partió con el *Mental Health and Technology Usage Dataset* (Kaggle). Se descartó porque
sus categorías se repartían en proporciones perfectamente parejas y un análisis publicado sobre ese
conjunto mostró coeficientes ≈ 0 para todas las variables de interés: la pregunta no tenía respuesta
posible con esos datos (decisiones 0.1–0.3 de `docs/bitacora_decisiones.md`).

**Objetivo general (Fase 1).** Definir la problemática y los objetivos del proyecto, y dejar
operativo y verificado un entorno de trabajo reproducible y versionado que sostenga las fases
siguientes.

**Objetivos específicos de esta fase.**
- Declarar el problema, las preguntas, el alcance, los supuestos, las restricciones y los criterios de éxito.
- Verificar que el cuaderno se ejecuta con el entorno del proyecto y con las dependencias declaradas.
- Verificar la estructura del repositorio y los artefactos que lo hacen reproducible.
- Probar el módulo `src/procesamiento.py` en casos normales, límite y de excepción.
- Documentar la procedencia del YRBS 2023 y evaluarla contra los criterios del curso.
- Dejar trazabilidad entre el mapa conceptual (v5), el repositorio y este cuaderno.

### Herramientas del ecosistema científico

| Pieza | Qué resuelve en este proyecto | Qué falla si se omite |
|---|---|---|
| Entorno aislado (`.venv` o entorno *conda*) | Fija las versiones de pandas/NumPy con las que se ejecutan F1 y F2. | Un compañero con otra versión de pandas obtiene tipos distintos al leer `q6orig`. |
| JupyterLab + `ipykernel` | Ejecuta los cuadernos con el intérprete del proyecto. | `ModuleNotFoundError` con un paquete que sí está instalado. |
| `requirements.txt` | Declara las dependencias exactas en la raíz. | `pip install -r requirements.txt` no instala nada y ningún cuaderno corre. |
| pandas / NumPy | Carga del CSV del CDC, diagnóstico de nulos, tablas de documentación. | — |
| `src/procesamiento.py` | Siete funciones del pipeline, compartidas por F1 y F2 sin copiar código. | La misma lógica se reescribe en cada cuaderno y diverge. |
| `pathlib` | Rutas relativas a la raíz del repositorio, iguales en Windows y macOS. | `C:/Users/...` funciona en un solo computador. |
| Git | Registra la historia y la autoría de cada cambio (**trazabilidad**). | No hay evidencia de contribución individual. |
| GitHub | Remoto compartido del grupo y evidencia evaluada. | El repositorio queda como depósito de archivos. |

In [1]:
import sys                       # intérprete en uso: se verifica en la sección 2
import os                        # variables de entorno (detección de conda)
import json                      # exportación de metadatos legibles
import inspect                   # listar las funciones del módulo del proyecto
import platform                  # sistema operativo, para la bitácora
import subprocess                # consultas a Git desde el cuaderno
import shutil                    # localizar el ejecutable de git sin suponer que existe
import importlib
import importlib.metadata as metadata   # versión instalada de cada paquete
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

# Reproducibilidad: misma semilla en todas las fases del proyecto.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 · Grupo 8 ·", date.today().isoformat())

Cuaderno de la Fase 1 · Grupo 8 · 2026-09-24


## 1. Definición del problema

**Qué hace este paso.** Deja el problema escrito como una **estructura de datos** y no como
párrafos sueltos. Problemática, pregunta, objetivos, alcance, supuestos, restricciones y
criterios de éxito quedan en el diccionario `PROYECTO`, del que después se derivan los metadatos
y las validaciones.

**Por qué así.** Un dato escrito una sola vez no puede contradecirse consigo mismo. Si cambia el
título o una cifra, cambia en un lugar.

> **Operacionalización de "uso de tecnología".** En este proyecto se mide con `q80`, la
> **frecuencia de uso de redes sociales** (8 niveles, de *"no uso redes sociales"* a *"más de una
> vez por hora"*). Es una escala de frecuencia, no de horas diarias.

In [2]:
PROYECTO = {
    "titulo": "Uso de redes sociales y salud mental autopercibida en adolescentes (YRBS 2023)",
    "grupo": "Grupo 8",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Matías Manríquez Ortiz", "Daniel Pérez Ramirez",
                    "Abigail Robles Chávez", "Roberto Sánchez Saldivia"],
    "repositorio": "https://github.com/MatiasManriquezO/proyecto-grupo8-mcdi500",
    "unidad_analisis": "Un estudiante de enseñanza media encuestado por el YRBS 2023 (CDC, EE. UU.)",
    "problematica": (
        "La adolescencia es una etapa en que suelen manifestarse problemas de salud mental, y "
        "coincide con un uso intensivo de redes sociales. La relación entre ambos es compleja: "
        "no se reduce a la depresión e interactúa con el sueño y la actividad física "
        "(Valkenburg et al., 2022). El problema es caracterizar esa relación con datos reales de "
        "una encuesta nacional con diseño muestral complejo, mediante un flujo reproducible."
    ),
    "pregunta_principal": (
        "¿Qué relación existe entre los patrones de uso de tecnología y los indicadores de salud "
        "mental autopercibida y horas de sueño en los estudiantes incluidos en el dataset YRBS 2023, "
        "considerando variables sociodemográficas y de actividad física?"
    ),
    "preguntas": [
        "¿Cómo se distribuyen las características sociodemográficas y los patrones de uso de redes "
        "sociales, salud mental autopercibida y sueño en la muestra?",
        "¿Cuál es el estado de calidad del YRBS 2023 (faltantes, duplicados, tipos, códigos fuera "
        "de dominio) y qué transformaciones se justifican?",
        "¿Existen patrones de asociación exploratorios entre uso de redes sociales, actividad "
        "física, salud mental y sueño?",
        "¿Cómo influyen conjuntamente esos factores en la salud mental autopercibida en un modelo "
        "ponderado por el diseño muestral? (Fases 3-4)",
    ],
    "objetivo_general": (
        "Analizar la relación entre los patrones de uso de tecnología —operacionalizados como "
        "frecuencia de uso de redes sociales (q80)— y los indicadores de salud mental autopercibida "
        "(q84) y horas de sueño (q85) en los estudiantes del YRBS 2023, controlando por variables "
        "sociodemográficas y de actividad física, mediante un flujo de trabajo reproducible y documentado."
    ),
    "objetivos_especificos": [
        "Caracterizar la distribución sociodemográfica y los patrones iniciales de uso de redes "
        "sociales, salud mental y sueño de la muestra mediante análisis exploratorio descriptivo.",
        "Diagnosticar y preprocesar el YRBS 2023 con un pipeline modular en Python "
        "(src/procesamiento.py), identificando y tratando faltantes, códigos fuera de dominio e "
        "inconsistencias de tipo.",
        "Validar el conjunto resultante con pruebas de casos normales, límite y de excepción, y "
        "exportarlo a F2/data/processed/ para las fases posteriores.",
        "Explorar asociaciones preliminares entre las variables de tecnología, estilo de vida y "
        "salud mental, preparando los datos para el modelamiento ponderado de las Fases 3 y 4.",
    ],
    "criterios_exito": [
        "El repositorio se clona, `pip install -r requirements.txt` instala las dependencias y los "
        "cuadernos F1 y F2 corren completos sin intervención manual.",
        "Las 11 columnas del subconjunto se seleccionan por código desde el archivo original.",
        "Cada decisión de preprocesamiento queda en la bitácora con la cifra que la respalda.",
        "Los cuatro integrantes tienen commits propios y atribuibles en el historial.",
    ],
    # Alcance, supuestos y restricciones van en categorías SEPARADAS y sin repetirse.
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible verificado",
                    "Selección y ficha del conjunto", "Pipeline de datos (F2)"],
        "excluye": ["Análisis ponderado e inferencia (F3)", "Modelamiento regresional (F3-F4)",
                    "Visualizaciones finales e informe de resultados (F4)"],
    },
    "supuestos": [
        "Las respuestas reflejan la percepción de cada estudiante, no un diagnóstico clínico.",
        "Los nulos de q1, q2, raceeth, q76, q80, q84 y q85 son no respuesta genuina y no saltos "
        "de pregunta: según el Apéndice C del codebook, ninguna depende de una pregunta previa.",
        "Las relaciones que se encuentren se interpretarán como asociaciones, no como causalidad.",
    ],
    "restricciones": [
        "Diseño muestral complejo: el YRBS trae factor de expansión (weight), estrato (stratum) y "
        "unidad primaria de muestreo (psu). En F1-F2 no se pondera, por lo que toda cifra describe "
        "a los 20.103 encuestados y no a la población de estudiantes de EE. UU. Las tres columnas "
        "se conservan para el análisis ponderado de F3 (decisión 2.6).",
        "Encuesta transversal de una sola aplicación: no hay variable de fecha ni análisis de evolución.",
        "El análisis usa 11 de las 117 columnas originales y ninguna fuente externa.",
    ],
}

La función siguiente presenta el proyecto. Recibe el diccionario como parámetro en lugar de leer
la variable global: así puede probarse de forma aislada y falla temprano si falta una clave.

In [3]:
def presentar_proyecto(config):
    '''
    Imprime la definición del proyecto de forma legible.

    Parámetros
    ----------
    config : dict
        Diccionario de configuración del proyecto.

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    '''
    obligatorias = ("titulo", "problematica", "pregunta_principal",
                    "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        raise KeyError(f"Faltan claves obligatorias en la configuración: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 70)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("Repositorio:", config["repositorio"])
    print("Unidad de análisis:", config["unidad_analisis"])
    print("\nProblemática\n ", config["problematica"])
    print("\nPregunta principal\n ", config["pregunta_principal"])
    print("\nPreguntas que orientan el trabajo")
    for p in config["preguntas"]:
        print("  ·", p)
    print("\nObjetivo general\n ", config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nCriterios de éxito")
    for c in config["criterios_exito"]:
        print("  ·", c)


presentar_proyecto(PROYECTO)

USO DE REDES SOCIALES Y SALUD MENTAL AUTOPERCIBIDA EN ADOLESCENTES (YRBS 2023)
MCDI500 · Programación para la Ciencia de Datos · Grupo 8
Repositorio: https://github.com/MatiasManriquezO/proyecto-grupo8-mcdi500
Unidad de análisis: Un estudiante de enseñanza media encuestado por el YRBS 2023 (CDC, EE. UU.)

Problemática
  La adolescencia es una etapa en que suelen manifestarse problemas de salud mental, y coincide con un uso intensivo de redes sociales. La relación entre ambos es compleja: no se reduce a la depresión e interactúa con el sueño y la actividad física (Valkenburg et al., 2022). El problema es caracterizar esa relación con datos reales de una encuesta nacional con diseño muestral complejo, mediante un flujo reproducible.

Pregunta principal
  ¿Qué relación existe entre los patrones de uso de tecnología y los indicadores de salud mental autopercibida y horas de sueño en los estudiantes incluidos en el dataset YRBS 2023, considerando variables sociodemográficas y de actividad f

El alcance, los supuestos y las restricciones se presentan aparte porque delimitan lo que **no**
se hará. La función comprueba además que ninguna afirmación aparezca repetida en dos categorías:
alcance, supuesto y restricción son cosas distintas y no deben fundirse.

In [4]:
def presentar_delimitacion(config):
    '''
    Imprime alcance, supuestos y restricciones, y verifica que no se repitan entre sí.

    Lanza
    -----
    AssertionError
        Si una misma afirmación aparece en más de una categoría.
    '''
    print("Alcance")
    print("  Incluye:", ", ".join(config["alcance"]["incluye"]))
    print("  Excluye:", ", ".join(config["alcance"]["excluye"]))
    for categoria in ("supuestos", "restricciones"):
        print(f"\n{categoria.capitalize()}")
        for linea in config[categoria]:
            print("  ·", linea)

    todas = (config["alcance"]["incluye"] + config["alcance"]["excluye"]
             + config["supuestos"] + config["restricciones"])
    normalizadas = [t.strip().lower() for t in todas]
    assert len(normalizadas) == len(set(normalizadas)), "Hay afirmaciones repetidas entre categorías."
    print(f"\n[OK] {len(todas)} afirmaciones de delimitación, sin repeticiones entre categorías.")


presentar_delimitacion(PROYECTO)

Alcance
  Incluye: Definición del problema, Entorno reproducible verificado, Selección y ficha del conjunto, Pipeline de datos (F2)
  Excluye: Análisis ponderado e inferencia (F3), Modelamiento regresional (F3-F4), Visualizaciones finales e informe de resultados (F4)

Supuestos
  · Las respuestas reflejan la percepción de cada estudiante, no un diagnóstico clínico.
  · Los nulos de q1, q2, raceeth, q76, q80, q84 y q85 son no respuesta genuina y no saltos de pregunta: según el Apéndice C del codebook, ninguna depende de una pregunta previa.
  · Las relaciones que se encuentren se interpretarán como asociaciones, no como causalidad.

Restricciones
  · Diseño muestral complejo: el YRBS trae factor de expansión (weight), estrato (stratum) y unidad primaria de muestreo (psu). En F1-F2 no se pondera, por lo que toda cifra describe a los 20.103 encuestados y no a la población de estudiantes de EE. UU. Las tres columnas se conservan para el análisis ponderado de F3 (decisión 2.6).
  · Encuesta

> **Sobre la restricción del diseño muestral.** Es la más importante del proyecto. Sin ponderar,
> una proporción calculada sobre la muestra no estima la proporción de la población, porque el
> YRBS sobrerrepresenta deliberadamente algunos estratos. Declararla ahora evita que en la Fase 4
> se interpreten cifras muestrales como si fueran poblacionales.

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba con evidencia que el cuaderno corre con el intérprete del
proyecto, desde el repositorio correcto, y que las dependencias declaradas están instaladas.

**Por qué es la primera comprobación.** Un paquete que "desaparece" o un `ModuleNotFoundError`
con una librería instalada no son errores de código, sino de entorno o de kernel. La función
reconoce tanto un `.venv` como un entorno *conda* (el grupo trabaja con Anaconda).

In [5]:
DEPENDENCIAS = ["pandas", "numpy", "ipykernel", "jupyterlab"]   # nombres en PyPI


def encontrar_raiz(desde=None):
    '''
    Sube desde la carpeta de trabajo hasta encontrar la raíz del repositorio (carpeta con .git).

    Lanza
    -----
    FileNotFoundError
        Si el cuaderno no se está ejecutando dentro del repositorio.
    '''
    actual = Path(desde or Path.cwd()).resolve()
    for carpeta in (actual, *actual.parents):
        if (carpeta / ".git").exists():
            return carpeta
    raise FileNotFoundError(f"No se encontró un repositorio Git sobre {actual}. "
                            "Abra JupyterLab dentro de proyecto-grupo8-mcdi500.")


def verificar_entorno(dependencias):
    '''
    Comprueba intérprete, tipo de entorno, carpeta de trabajo y versiones de las librerías.

    Retorna
    -------
    dict
        Resultado de cada comprobación, para dejarlo en los metadatos de la fase.
    '''
    reporte = {}
    ejecutable = Path(sys.executable)
    es_venv = sys.prefix != sys.base_prefix
    env_conda = os.environ.get("CONDA_DEFAULT_ENV")
    es_conda = (Path(sys.prefix) / "conda-meta").exists()
    if es_venv:
        tipo = "entorno virtual (venv)"
    elif es_conda:
        tipo = f"entorno conda '{env_conda or Path(sys.prefix).name}'"
    else:
        tipo = "Python del sistema"
    reporte.update(interprete=str(ejecutable), tipo_entorno=tipo,
                   entorno_aislado=bool(es_venv or es_conda))

    print("Intérprete        :", ejecutable)
    print("Tipo de entorno   :", tipo, "[OK]" if reporte["entorno_aislado"] else "[AVISO]")
    if env_conda == "base":
        print("                    [AVISO] es el entorno base de Anaconda; se recomienda uno propio del proyecto")
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema           :", platform.system(), platform.release())
    print("Python            :", sys.version.split()[0])
    reporte["python"] = sys.version.split()[0]

    print("\nDependencias declaradas")
    versiones = {}
    for paquete in dependencias:
        try:
            versiones[paquete] = metadata.version(paquete)
            print(f"  [OK]    {paquete:12} {versiones[paquete]}")
        except metadata.PackageNotFoundError:
            versiones[paquete] = None
            print(f"  [FALTA] {paquete:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


RAIZ = encontrar_raiz()
print("Raíz del repositorio:", RAIZ, "\n")
ENTORNO = verificar_entorno(DEPENDENCIAS)

Raíz del repositorio: C:\Users\danie\OneDrive\Documents\GitHub\proyecto-grupo8-mcdi500 

Intérprete        : C:\Users\danie\.conda\envs\notebook-env\python.exe
Tipo de entorno   : entorno conda 'notebook-env' [OK]
Carpeta de trabajo: C:\Users\danie\OneDrive\Documents\GitHub\proyecto-grupo8-mcdi500\F1\notebooks
Sistema           : Windows 11
Python            : 3.14.7

Dependencias declaradas
  [OK]    pandas       3.0.6
  [OK]    numpy        2.5.3
  [OK]    ipykernel    7.3.0
  [OK]    jupyterlab   4.5.9


**Cómo se lee este resultado.**

| Lo que ve | Qué significa | Qué hacer |
| --- | --- | --- |
| Tipo de entorno = Python del sistema | El kernel no es el del proyecto | *Kernel → Change Kernel* |
| Entorno conda `base` | Funciona, pero comparte librerías con todo lo demás | Crear un entorno propio del proyecto |
| Una librería aparece como `[FALTA]` | Se instaló fuera del entorno | Activar el entorno y reinstalar |
| No se encuentra la raíz del repositorio | JupyterLab se abrió fuera del proyecto | Abrirlo desde `proyecto-grupo8-mcdi500/` |

## 3. Estructura del repositorio y artefactos

**Qué hace este paso.** Verifica que existan las carpetas comprometidas en el mapa conceptual
(bloque 7) y que los cuadernos F1 y F2 lean los datos desde la **misma** ruta.

**Decisión de estructura (documentada).** Los datos originales viven en `F1/data/raw/` y la salida
del pipeline en `F2/data/processed/`; `src/` y `docs/` están en la raíz. Es la estructura del mapa
conceptual. La condición para sostenerla es que ningún cuaderno suponga otra ruta: la última
comprobación de esta celda lo verifica leyendo el cuaderno de la Fase 2.

In [6]:
DIR_CRUDO = RAIZ / "F1" / "data" / "raw"            # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "F2" / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                            # documentación general del proyecto (bitácora)
DIR_SALIDA_F1 = RAIZ / "F1" / "data" / "docs"       # salidas de este cuaderno: diccionario, tablas, metadatos
DIR_SALIDA_F1.mkdir(parents=True, exist_ok=True)    # se crea si no existe
DIR_SRC = RAIZ / "src"                              # módulo reutilizable del proyecto
ARCHIVO = DIR_CRUDO / "XXH2023_YRBSS_data.csv"
NB_F2 = RAIZ / "F2" / "notebooks" / "S1_F2_Preprocesamiento.ipynb"

ESTRUCTURA = [DIR_CRUDO, RAIZ / "F1" / "notebooks", DIR_PROCESADO, RAIZ / "F2" / "notebooks",
              RAIZ / "F3" / "notebooks", RAIZ / "F4" / "notebooks", DIR_DOCS, DIR_SALIDA_F1, DIR_SRC]

faltan = [p.relative_to(RAIZ).as_posix() for p in ESTRUCTURA if not p.is_dir()]
assert not faltan, f"Faltan carpetas declaradas en el mapa conceptual: {faltan}"
print(f"[OK] Las {len(ESTRUCTURA)} carpetas declaradas existen.\n")

# Árbol de dos niveles, sin carpetas internas de Git ni de Jupyter.
OMITIR = {".git", ".ipynb_checkpoints", "__pycache__", ".venv"}
for ruta in sorted(RAIZ.rglob("*")):
    rel = ruta.relative_to(RAIZ)
    if OMITIR & set(rel.parts) or len(rel.parts) > 3:
        continue
    sangria = "    " * (len(rel.parts) - 1)
    print(f"  {sangria}{rel.name}{'/' if ruta.is_dir() else ''}")

# Coherencia de rutas entre cuadernos: F2 debe leer el archivo desde F1/data/raw.
fuente_f2 = "".join("".join(c["source"]) for c in json.loads(NB_F2.read_text(encoding="utf-8"))["cells"])
assert "'F1' / 'data' / 'raw'" in fuente_f2, "El cuaderno F2 no lee desde F1/data/raw."
assert "'F2' / 'data' / 'raw'" not in fuente_f2, "El cuaderno F2 apunta a F2/data/raw, que no existe."
print("\n[OK] F1 y F2 usan la misma ruta de datos: F1/data/raw/XXH2023_YRBSS_data.csv")

[OK] Las 9 carpetas declaradas existen.

  .gitignore
  docs/
      bitacora_decisiones.md
      diccionario_variables.md
      diccionario_variables_f1.csv
      diccionario_variables_f2.csv
      evaluacion_criterios_dataset.csv
      Informe/
          f1_s01_evaluacion_entregable_grupo8.pdf
      Mapa Conceptual Proyecto/
          mapa_conceptual_grupo8_v5.png
      metadatos_f2.csv
      metadatos_fase1.json
      registro_preprocesamiento_f2.csv
      vinculacion_mapa_conceptual.csv
  F1/
      data/
          docs/
          raw/
      notebooks/
          S1_F1_Definicion.ipynb
  F2/
      data/
          processed/
      docs/
          diccionario_variables_f2.csv
          metadatos_f2.csv
          registro_preprocesamiento_f2.csv
      notebooks/
          S1_F2_Preprocesamiento.ipynb
  F3/
      notebooks/
          S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb
  F4/
      notebooks/
          Fase 4.md
  README.md
  requirements.txt
  src/
      procesamiento.py

[OK] F1

### 3.1 El archivo `.gitignore`

El repositorio ya tiene un `.gitignore`, así que el cuaderno **no lo sobrescribe**: lo verifica.
Deben quedar fuera lo que se regenera (entorno virtual, cachés), lo local de cada equipo (archivos
del sistema operativo) y lo secreto. También se comprueba lo contrario: que el **dato original no
esté ignorado**, porque sin él nadie puede reproducir el análisis.

In [7]:
def verificar_gitignore(raiz, obligatorias=(".venv/", "__pycache__/", ".ipynb_checkpoints/")):
    '''
    Comprueba que .gitignore exista, contenga las reglas mínimas y no ignore el dato original.

    Retorna
    -------
    list of str
        Reglas activas (sin comentarios ni líneas vacías).
    '''
    archivo = raiz / ".gitignore"
    assert archivo.exists(), "Falta .gitignore en la raíz."
    reglas = [l.strip() for l in archivo.read_text(encoding="utf-8").splitlines()
              if l.strip() and not l.strip().startswith("#")]
    for regla in obligatorias:
        assert regla in reglas, f".gitignore no excluye {regla}"
    print(f"[OK] .gitignore con {len(reglas)} reglas activas; incluye {', '.join(obligatorias)}")

    # git check-ignore devuelve código 0 si el archivo SÍ está ignorado.
    if shutil.which("git"):
        ignorado = subprocess.run(["git", "check-ignore", "-q", str(ARCHIVO)], cwd=raiz).returncode == 0
        assert not ignorado, "El dato original está ignorado: nadie podría reproducir el análisis."
        print("[OK] El dato original F1/data/raw/ NO está ignorado: se versiona.")
    return reglas


reglas_gitignore = verificar_gitignore(RAIZ)

[OK] .gitignore con 31 reglas activas; incluye .venv/, __pycache__/, .ipynb_checkpoints/
[OK] El dato original F1/data/raw/ NO está ignorado: se versiona.


### 3.2 El archivo `requirements.txt`

Declarar las dependencias separa "funciona en mi computador" de un entorno reproducible. El
grupo genera este archivo con `pip freeze` (decisión 3.7 de la bitácora), que incluye también las
dependencias indirectas. El cuaderno comprueba tres cosas: que **no esté vacío**, que fije las
librerías del proyecto y que las versiones fijadas coincidan con las que están realmente
instaladas en este entorno.

In [8]:
def verificar_requirements(ruta, versiones_instaladas):
    '''
    Lee requirements.txt y lo contrasta con las versiones instaladas.

    Parámetros
    ----------
    ruta : Path
        Archivo requirements.txt.
    versiones_instaladas : dict
        Paquete -> versión detectada en el entorno (o None).

    Retorna
    -------
    pd.DataFrame
        Una fila por dependencia del proyecto: versión fijada, instalada y si coinciden.

    Lanza
    -----
    ValueError
        Si el archivo está vacío o no fija pandas y numpy.
    '''
    lineas = [l.strip() for l in ruta.read_text(encoding="utf-8").splitlines()
              if l.strip() and not l.startswith("#")]
    if not lineas:
        raise ValueError("requirements.txt está vacío: pip install -r no instalaría nada.")
    fijadas = {}
    for l in lineas:
        if "==" in l:
            nombre, version = l.split("==", 1)
            fijadas[nombre.strip().lower().replace("_", "-")] = version.strip()
    for obligatoria in ("pandas", "numpy"):
        if obligatoria not in fijadas:
            raise ValueError(f"requirements.txt no fija {obligatoria}.")

    filas = []
    for paquete, instalada in versiones_instaladas.items():
        fijada = fijadas.get(paquete.lower())
        filas.append({"paquete": paquete, "fijada": fijada, "instalada": instalada,
                      "coincide": "sí" if fijada == instalada else "NO"})
    print(f"requirements.txt: {len(lineas)} líneas, {len(fijadas)} paquetes con versión fijada.")
    return pd.DataFrame(filas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
tabla_requirements = verificar_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
tabla_requirements

requirements.txt: 97 líneas, 97 paquetes con versión fijada.


,paquete,fijada,instalada,coincide
0,pandas,3.0.6,3.0.6,sí
1,numpy,2.5.3,2.5.3,sí
2,ipykernel,7.3.0,7.3.0,sí
3,jupyterlab,4.6.4,4.5.9,NO


> **Cómo se lee.** Una fila con `coincide = NO` no rompe el cuaderno, pero indica que este
> computador no tiene el entorno declarado: se corrige con `python -m pip install -r requirements.txt`
> dentro del entorno del proyecto, o regenerando el archivo con `pip freeze > requirements.txt` si
> el grupo decide actualizar la versión.

## 4. Módulo del proyecto: `src/procesamiento.py`

**Qué hace este paso.** Importa el módulo que comparten F1 y F2 y lista sus funciones con la
primera línea de su documentación.

**Por qué.** La Unidad 1 pide funciones, excepciones **y módulos**. Un cuaderno documenta y
comunica; el módulo permite que la misma función se use en todas las fases sin copiarla. El
cuaderno de la Fase 2 importa exactamente estas funciones.

In [9]:
if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))

importlib.invalidate_caches()
import procesamiento
importlib.reload(procesamiento)          # por si el cuaderno se vuelve a ejecutar

print("Importado:", procesamiento.__name__, "desde", Path(procesamiento.__file__).relative_to(RAIZ).as_posix(), "\n")
funciones = [(nombre, (inspect.getdoc(f) or "").splitlines()[0])
             for nombre, f in inspect.getmembers(procesamiento, inspect.isfunction)
             if f.__module__ == procesamiento.__name__]
pd.DataFrame(funciones, columns=["función", "propósito"])

Importado: procesamiento desde src/procesamiento.py 



,función,propósito
0,cargar_datos,Carga el CSV del YRBS 2023 tal como viene del ...
1,clasificar_variables,Clasifica cada variable según su rol metodológ...
2,diagnosticar_datos,"Diagnóstico de calidad: nulos, duplicados y ti..."
3,guardar_dataset,"Guarda el DataFrame procesado, creando directo..."
4,seleccionar_columnas,Selecciona y renombra las columnas definidas e...
5,transformar_datos,Convierte variables ordinales a pd.Categorical...
6,validar_datos,Valida condiciones mínimas del DataFrame proce...
7,verificar_dominios,Detecta códigos inválidos antes de que Categor...


### 4.1 Pruebas del módulo (casos normal, límite y excepción)

Las pruebas usan un **DataFrame sintético de tres filas** con los nombres de columna del archivo
original, de modo que no dependen de tener descargado el CSV y corren en segundos. `assert` detiene
el cuaderno si una condición no se cumple, dejando constancia de la falla en la salida.

In [10]:
def muestra_sintetica(n=3):
    '''DataFrame mínimo con las columnas originales que usa el pipeline y códigos válidos.'''
    base = {c: [1.0] * n for c in procesamiento.COLUMNAS_ANALISIS}
    base["record"] = list(range(1, n + 1))                  # identificador único
    base["weight"] = [0.86, 0.89, 0.51][:n]
    base["q84"] = [1.0, 3.0, 5.0][:n]                       # salud mental: Never / Sometimes / Always
    return pd.DataFrame(base)


# --- Caso normal -----------------------------------------------------------
sel = procesamiento.seleccionar_columnas(muestra_sintetica())
assert sel.shape == (3, 11), f"Se esperaban 3 x 11 y se obtuvo {sel.shape}"
assert list(sel.columns) == list(procesamiento.COLUMNAS_ANALISIS.values())
assert all(v == {} for v in procesamiento.verificar_dominios(sel).values())
assert procesamiento.validar_datos(sel)["validacion_global"]
print("[OK] Caso normal: selección de 11 columnas renombradas, dominios válidos, validación global verdadera")

# --- Caso límite 1: DataFrame sin filas -------------------------------------
vacio = procesamiento.seleccionar_columnas(muestra_sintetica().head(0))
assert vacio.shape == (0, 11)
print("[OK] Caso límite: DataFrame vacío conserva sus 11 columnas y no falla")

# --- Caso límite 2: un código fuera del codebook ----------------------------
fuera = muestra_sintetica()
fuera.loc[0, "q84"] = 9.0                                # q84 solo admite 1..5
errores = procesamiento.verificar_dominios(procesamiento.seleccionar_columnas(fuera))
assert errores["salud_mental_cod"] == {"9.0": 1}, errores["salud_mental_cod"]
print("[OK] Caso límite: el código 9 en q84 se detecta antes de convertir a Categorical:", errores["salud_mental_cod"])

# --- Caso límite 3: falta una columna en el archivo -------------------------
# seleccionar_columnas omite en silencio la columna ausente; validar_datos debe detectarlo.
sin_q80 = procesamiento.seleccionar_columnas(muestra_sintetica().drop(columns="q80"))
assert sin_q80.shape[1] == 10
assert not procesamiento.validar_datos(sin_q80)["columnas_obligatorias_presentes"]
print("[OK] Caso límite: sin q80 se seleccionan 10 columnas y validar_datos lo marca como falla")

# --- Excepción 1: archivo inexistente --------------------------------------
try:
    procesamiento.cargar_datos(str(DIR_CRUDO / "no_existe.csv"))
    raise AssertionError("cargar_datos debió lanzar FileNotFoundError")
except FileNotFoundError as e:
    print(f"[OK] Excepción capturada (archivo inexistente): {type(e).__name__}")

# --- Excepción 2: columna ausente en la verificación de dominios -----------
try:
    procesamiento.verificar_dominios(sin_q80)
    raise AssertionError("verificar_dominios debió lanzar ValueError")
except ValueError as e:
    print(f"[OK] Excepción capturada (columna ausente): {e}")

[OK] Caso normal: selección de 11 columnas renombradas, dominios válidos, validación global verdadera
[OK] Caso límite: DataFrame vacío conserva sus 11 columnas y no falla
[OK] Caso límite: el código 9 en q84 se detecta antes de convertir a Categorical: {'9.0': 1}
[OK] Caso límite: sin q80 se seleccionan 10 columnas y validar_datos lo marca como falla
[OK] Excepción capturada (archivo inexistente): FileNotFoundError
[OK] Excepción capturada (columna ausente): Falta columna redes_sociales_cod


> **Hallazgo de las pruebas.** El caso límite 3 muestra que `seleccionar_columnas` no falla si
> falta una columna: la omite. No es un error, porque `validar_datos` lo detecta después, pero
> significa que **la validación no es opcional**: si el pipeline se ejecutara sin ella, un archivo
> incompleto pasaría sin aviso.

## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta la procedencia del YRBS 2023, la evalúa contra los criterios de
selección del curso y declara el rol analítico de cada variable del proyecto.

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | 2023 National Youth Risk Behavior Survey (YRBS) |
| Institución | Centers for Disease Control and Prevention (CDC), EE. UU. |
| Enlace | https://www.cdc.gov/yrbs/data/index.html |
| Documentación | *2023 YRBS Data User's Guide* (codebook, págs. 21–55; Apéndice C) |
| Archivo en el repositorio | `F1/data/raw/XXH2023_YRBSS_data.csv` |
| Estructura esperada | 20.103 filas × 117 columnas |
| Unidad de observación | Un estudiante encuestado |
| Tipo de estudio | Encuesta transversal con diseño muestral complejo (`weight`, `stratum`, `psu`) |
| Licencia | Dato público de agencia federal de EE. UU., de libre uso con atribución |

**Referencias en APA 7:**

> Centers for Disease Control and Prevention. (2024). *2023 Youth Risk Behavior Survey data* [Conjunto de datos]. https://www.cdc.gov/yrbs/data/index.html

> Centers for Disease Control and Prevention. (2024). *2023 YRBS data user's guide*. https://www.cdc.gov/yrbs/media/pdf/2023/2023_National_YRBS_Data_Users_Guide508.pdf

La ficha declara el **rol analítico** de las 117 columnas del archivo original. Los roles se
asignan por lista explícita y el conteo se calcula a partir de ellas, de modo que la suma debe dar
exactamente 117. Las preguntas `q` que no están en ninguna lista son escalas de respuesta
codificada y se cuentan como ordinales.

In [11]:
# Las 117 columnas del archivo, en el orden del CDC.
COLUMNAS_ORIGINALES = (["site", "raceeth", "q6orig", "q7orig", "record", "orig_rec"]
                       + [f"q{i}" for i in range(1, 108)]
                       + ["BMIPCT", "weight", "stratum", "psu"])

ROLES_ARCHIVO = {
    "identificador":     ["record", "orig_rec"],
    "diseno_muestral":   ["stratum", "psu"],
    "continua":          ["q6", "q7", "BMIPCT", "weight"],   # estatura (m), peso (kg), percentil IMC, factor de expansión
    "discreta":          ["q75", "q76", "q77"],             # conteos de días en la semana
    "binaria":           ["q2", "q4", "q18", "q19", "q24", "q25", "q26", "q27", "q28", "q31",
                          "q35", "q56", "q88", "q100", "q101", "q102", "q105", "q106"],
    "nominal":           ["site", "raceeth", "q37", "q45", "q62", "q64", "q67", "q86"],
    "alta_cardinalidad": ["q5", "q6orig", "q7orig"],        # texto: respuesta múltiple y valores digitados
    "fecha":             [],
}


def contar_roles(columnas, roles):
    '''
    Cuenta columnas por rol analítico; las q no asignadas se cuentan como ordinales.

    Lanza
    -----
    ValueError
        Si una columna aparece en dos roles o si un rol nombra una columna inexistente.
    '''
    asignadas = [c for lista in roles.values() for c in lista]
    repetidas = {c for c in asignadas if asignadas.count(c) > 1}
    if repetidas:
        raise ValueError(f"Columnas con dos roles: {repetidas}")
    inexistentes = set(asignadas) - set(columnas)
    if inexistentes:
        raise ValueError(f"Roles con columnas que no están en el archivo: {inexistentes}")
    conteo = {rol: len(lista) for rol, lista in roles.items()}
    conteo["ordinal"] = len([c for c in columnas if c not in asignadas])
    return conteo


FICHA = {
    "titulo": "2023 National Youth Risk Behavior Survey (YRBS)",
    "autor": "Centers for Disease Control and Prevention (CDC)",
    "plataforma": "cdc.gov",
    "url": "https://www.cdc.gov/yrbs/data/index.html",
    "licencia_abierta": True,
    "unidad_observacion": "Un estudiante encuestado",
    "filas": 20103,
    "columnas": len(COLUMNAS_ORIGINALES),
    "roles": contar_roles(COLUMNAS_ORIGINALES, ROLES_ARCHIVO),
    "pct_nulos_max_variable": 100.0,        # orig_rec: vacía (decisión 2.1)
    "pct_nulos_max_con_datos": 47.4,        # q105, la variable con datos más incompleta
}

assert sum(FICHA["roles"].values()) == FICHA["columnas"] == 117
print(json.dumps(FICHA["roles"], indent=2, ensure_ascii=False))

{
  "identificador": 2,
  "diseno_muestral": 2,
  "continua": 4,
  "discreta": 3,
  "binaria": 18,
  "nominal": 8,
  "alta_cardinalidad": 3,
  "fecha": 0,
  "ordinal": 77
}


### 5.2 Evaluación contra los criterios del curso

La función compara la ficha con los mínimos del proyecto y devuelve el veredicto por criterio.
No decide por el grupo: expone lo que falta.

In [12]:
CRITERIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoría sin grupos vacíos"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analíticos"),
    "continua": ("Numéricas continuas", 2, "Escalamiento y valores atípicos"),
    "discreta": ("Numérica discreta", 1, "Obliga a decidir número frente a categoría"),
    "nominal": ("Categóricas nominales", 2, "Codificación One-Hot"),
    "ordinal_o_binaria": ("Binaria u ordinal", 1, "Codificación con orden declarado"),
    "fecha": ("Fecha", 1, "Parseo y derivación de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalización y agrupación"),
}


def evaluar_criterios(ficha, criterios=CRITERIOS):
    '''
    Compara una ficha de conjunto de datos con los mínimos exigidos por el curso.

    Retorna
    -------
    pd.DataFrame
        Una fila por criterio, con el valor observado y el veredicto.

    Lanza
    -----
    KeyError
        Si la ficha no declara los roles analíticos.
    '''
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analítico.")
    roles = ficha["roles"]
    observado = {
        "filas": ficha.get("filas", 0),
        "columnas": ficha.get("columnas", 0),
        "continua": roles.get("continua", 0),
        "discreta": roles.get("discreta", 0),
        "nominal": roles.get("nominal", 0),
        "ordinal_o_binaria": roles.get("ordinal", 0) + roles.get("binaria", 0),
        "fecha": roles.get("fecha", 0),
        "alta_cardinalidad": roles.get("alta_cardinalidad", 0),
    }
    filas = []
    for clave, (etiqueta, minimo, motivo) in criterios.items():
        valor = observado[clave]
        filas.append({"criterio": etiqueta, "mínimo": minimo, "observado": valor,
                      "cumple": "sí" if valor >= minimo else "NO", "por qué se pide": motivo})

    con_datos = ficha.get("pct_nulos_max_con_datos", 0.0)
    filas.append({"criterio": "Alguna variable con ≥1% de nulos", "mínimo": 1.0,
                  "observado": con_datos, "cumple": "sí" if con_datos >= 1.0 else "NO",
                  "por qué se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar"})
    maximo = ficha.get("pct_nulos_max_variable", 0.0)
    filas.append({"criterio": "Ninguna variable sobre 60% de nulos", "mínimo": 60.0,
                  "observado": maximo, "cumple": "sí" if maximo <= 60.0 else "NO",
                  "por qué se pide": "Por encima de eso la variable no es recuperable"})
    tabla = pd.DataFrame(filas)
    for col in ("mínimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion = evaluar_criterios(FICHA)
evaluacion

,criterio,mínimo,observado,cumple,por qué se pide
0,Filas,2000,20103,sí,Permite agrupar por categoría sin grupos vacíos
1,Columnas,12,117,sí,Asegura variedad de roles analíticos
2,Numéricas continuas,2,4,sí,Escalamiento y valores atípicos
3,Numérica discreta,1,3,sí,Obliga a decidir número frente a categoría
4,Categóricas nominales,2,8,sí,Codificación One-Hot
5,Binaria u ordinal,1,95,sí,Codificación con orden declarado
6,Fecha,1,0,NO,Parseo y derivación de variables temporales
7,Texto o alta cardinalidad,1,3,sí,Normalización y agrupación
8,Alguna variable con ≥1% de nulos,1,47.4,sí,Un conjunto perfecto no tiene preprocesamiento...
9,Ninguna variable sobre 60% de nulos,60,100,NO,Por encima de eso la variable no es recuperable


El archivo original incumple dos criterios. Se evalúa entonces el **conjunto de trabajo**: el
mismo archivo después de aplicar la única eliminación ya documentada, `orig_rec` (100 % nula,
decisión 2.1). Así queda a la vista qué incumplimiento se resuelve con una decisión de
preprocesamiento y cuál es propio del diseño de la encuesta.

In [13]:
roles_trabajo = {rol: [c for c in cols if c != "orig_rec"] for rol, cols in ROLES_ARCHIVO.items()}
FICHA_TRABAJO = {**FICHA,
                 "columnas": FICHA["columnas"] - 1,
                 "roles": contar_roles([c for c in COLUMNAS_ORIGINALES if c != "orig_rec"], roles_trabajo),
                 "pct_nulos_max_variable": FICHA["pct_nulos_max_con_datos"]}

evaluacion_trabajo = evaluar_criterios(FICHA_TRABAJO)
comparacion = evaluacion[["criterio", "cumple"]].rename(columns={"cumple": "archivo original"})
comparacion["conjunto de trabajo"] = evaluacion_trabajo["cumple"]

incumplidos = evaluacion_trabajo.loc[evaluacion_trabajo["cumple"] == "NO", "criterio"].tolist()
print(f"Criterios evaluados : {len(evaluacion_trabajo)}")
print(f"Criterios cumplidos : {len(evaluacion_trabajo) - len(incumplidos)}")
print("No cumple           :", incumplidos or "ninguno")
comparacion

Criterios evaluados : 10
Criterios cumplidos : 9
No cumple           : ['Fecha']


,criterio,archivo original,conjunto de trabajo
0,Filas,sí,sí
1,Columnas,sí,sí
2,Numéricas continuas,sí,sí
3,Numérica discreta,sí,sí
4,Categóricas nominales,sí,sí
5,Binaria u ordinal,sí,sí
6,Fecha,NO,NO
7,Texto o alta cardinalidad,sí,sí
8,Alguna variable con ≥1% de nulos,sí,sí
9,Ninguna variable sobre 60% de nulos,NO,sí


> **Lectura del resultado.** El YRBS 2023 cumple todos los criterios del curso salvo uno: **no
> tiene variable de fecha**. No es un descuido de la ficha. El YRBS es una encuesta **transversal**
> aplicada una sola vez, y el CDC no publica la fecha de respuesta de cada estudiante para
> resguardar el anonimato. Se declara como restricción en la sección 1 y como decisión abierta en
> la sección 10. El criterio de texto y alta cardinalidad sí se cumple con datos reales: `q5`
> (raza, respuesta múltiple, 35 combinaciones distintas) y `q6orig`/`q7orig` (estatura y peso tal
> como se digitaron, con valores como `"N N"`).

### 5.3 Diccionario de variables del proyecto

El diccionario declara el **rol analítico** de cada una de las 11 columnas del subconjunto. No se
escribe a mano: el nombre y el rol se leen de `src/procesamiento.py` (`COLUMNAS_ANALISIS` y
`clasificar_variables`), y el cuaderno solo agrega la pregunta y la escala del codebook. Así el
diccionario no puede contradecir al código que ejecuta la Fase 2.

> **Decisión de rol (bitácora 2.4).** `q1`, `q76`, `q80`, `q84` y `q85` vienen codificadas como
> enteros, pero el grupo las trata como **ordinales**, porque la distancia entre categorías como
> *"Rara vez"* y *"A veces"* no está garantizada. `q76` (días activos) es el caso límite: sus
> categorías sí son equidistantes (un día cada una), y en el archivo original se cuenta como
> discreta. Se deja registrada como decisión abierta.

In [14]:
CODEBOOK = {
    "record":  ("Identificador del registro", "entero único"),
    "weight":  ("Factor de expansión", "continuo > 0"),
    "stratum": ("Estrato del diseño muestral", "código"),
    "psu":     ("Unidad primaria de muestreo", "código"),
    "q1":      ("How old are you?", "1 = ≤12 años … 7 = ≥18 años"),
    "q2":      ("What is your sex?", "1 = Female, 2 = Male"),
    "raceeth": ("Raza/etnicidad (combina q4 y q5, CDC)", "8 categorías"),
    "q84":     ("During the past 30 days, how often was your mental health not good?", "1 = Never … 5 = Always"),
    "q80":     ("How often do you use social media?", "1 = no uso … 8 = más de una vez por hora"),
    "q85":     ("On an average school night, how many hours of sleep do you get?", "1 = ≤4 h … 7 = ≥10 h"),
    "q76":     ("Past 7 days, days physically active ≥60 minutes", "1 = 0 días … 8 = 7 días"),
}
ROL_EN_ESTUDIO = {"q84": "desenlace", "q80": "exposición", "q85": "control", "q76": "control",
                  "q1": "control demográfico", "q2": "control demográfico",
                  "raceeth": "control demográfico", "record": "identificador",
                  "weight": "diseño muestral", "stratum": "diseño muestral", "psu": "diseño muestral"}
PCT_NULOS = {"record": 0.0, "weight": 0.0, "stratum": 0.0, "psu": 0.0, "q1": 0.5, "q2": 0.8,
             "raceeth": 1.8, "q76": 6.1, "q85": 13.2, "q84": 21.9, "q80": 24.4}   # bitácora, Fase 2

# El rol analítico se lee del módulo: se le pasa un DataFrame vacío con los nombres renombrados.
roles_modulo = procesamiento.clasificar_variables(
    pd.DataFrame(columns=list(procesamiento.COLUMNAS_ANALISIS.values())))

diccionario = pd.DataFrame([
    {"variable_original": original, "nombre_proyecto": nuevo,
     "rol_analitico": roles_modulo[nuevo]["rol_equipo"], "rol_en_estudio": ROL_EN_ESTUDIO[original],
     "pregunta": CODEBOOK[original][0], "escala": CODEBOOK[original][1],
     "pct_nulos": PCT_NULOS[original]}
    for original, nuevo in procesamiento.COLUMNAS_ANALISIS.items()
])

# Verificación: el diccionario cubre exactamente las columnas que selecciona el módulo.
assert set(diccionario["variable_original"]) == set(procesamiento.COLUMNAS_ANALISIS)
assert set(diccionario["variable_original"]) <= set(COLUMNAS_ORIGINALES)
print(diccionario["rol_analitico"].value_counts().to_string(), "\n")
diccionario

rol_analitico
ordinal               5
nominal               2
identificador         1
muestral (weight)     1
muestral (stratum)    1
muestral (psu)        1 



,variable_original,nombre_proyecto,rol_analitico,rol_en_estudio,pregunta,escala,pct_nulos
0,record,id_registro,identificador,identificador,Identificador del registro,entero único,0.0
1,weight,peso_muestral,muestral (weight),diseño muestral,Factor de expansión,continuo > 0,0.0
2,stratum,estrato,muestral (stratum),diseño muestral,Estrato del diseño muestral,código,0.0
3,psu,psu,muestral (psu),diseño muestral,Unidad primaria de muestreo,código,0.0
4,q1,edad_cod,ordinal,control demográfico,How old are you?,1 = ≤12 años … 7 = ≥18 años,0.5
5,q2,sexo_cod,nominal,control demográfico,What is your sex?,"1 = Female, 2 = Male",0.8
6,raceeth,raceeth_cod,nominal,control demográfico,"Raza/etnicidad (combina q4 y q5, CDC)",8 categorías,1.8
7,q84,salud_mental_cod,ordinal,desenlace,"During the past 30 days, how often was your me...",1 = Never … 5 = Always,21.9
8,q80,redes_sociales_cod,ordinal,exposición,How often do you use social media?,1 = no uso … 8 = más de una vez por hora,24.4
9,q85,sueno_cod,ordinal,control,"On an average school night, how many hours of ...",1 = ≤4 h … 7 = ≥10 h,13.2


> **Los nulos de `q80` y `q84` (24,4 % y 21,9 %) no son saltos de pregunta.** Según el Apéndice C
> del codebook, ninguna de las variables de análisis depende de una pregunta previa: sus nulos son
> no respuesta genuina. Declararlo aquí evita que en la Fase 2 se imputen por inercia.

### 5.4 Comprobación del archivo (si ya está descargado)

Esta celda no descarga ni modifica nada. Si el archivo está en `F1/data/raw/`, contrasta su
estructura real con la ficha y separa explícitamente las dos cifras que no deben confundirse: el
**archivo original** (20.103 × 117) y el **subconjunto de trabajo** (20.103 × 11). Si el archivo no
está, informa qué falta y el cuaderno sigue: la Fase 1 no depende de él.

In [15]:
if ARCHIVO.exists():
    df_crudo = procesamiento.cargar_datos(str(ARCHIVO))
    subconjunto = procesamiento.seleccionar_columnas(df_crudo)

    print("Archivo            :", ARCHIVO.relative_to(RAIZ).as_posix())
    print("Archivo original   :", df_crudo.shape, "(filas, columnas)")
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    print("Subconjunto (F2)   :", subconjunto.shape, "— mismas filas, 11 columnas")
    assert df_crudo.shape == (FICHA["filas"], FICHA["columnas"]), "Ficha y archivo difieren."
    assert list(df_crudo.columns) == COLUMNAS_ORIGINALES, "El orden o los nombres de columna difieren."

    pct = df_crudo.isna().mean().mul(100).round(1)
    print("\nNulos: variable más incompleta          :", pct.idxmax(), f"{pct.max()}%")
    print("       más incompleta con datos          :", pct[pct < 100].idxmax(), f"{pct[pct < 100].max()}%")
    print("Duplicados (filas completas)             :", int(df_crudo.duplicated().sum()))
    print("record es identificador único            :", df_crudo["record"].is_unique)
    print("\nValores distintos en las columnas de texto/alta cardinalidad:")
    print(df_crudo[ROLES_ARCHIVO["alta_cardinalidad"]].nunique().to_string())
    print("\nNulos del subconjunto (%):")
    print(subconjunto.isna().mean().mul(100).round(1).sort_values(ascending=False).to_string())
else:
    df_crudo = None
    print("[PENDIENTE] El archivo aún no está en", DIR_CRUDO.relative_to(RAIZ).as_posix())
    print("Descárguelo desde:", FICHA["url"])
    print("\nLa Fase 1 no requiere el archivo: la exploración es trabajo de la Fase 2.")

Archivo            : F1/data/raw/XXH2023_YRBSS_data.csv
Archivo original   : (20103, 117) (filas, columnas)
Declarado en ficha : (20103, 117)
Subconjunto (F2)   : (20103, 11) — mismas filas, 11 columnas

Nulos: variable más incompleta          : orig_rec 100.0%
       más incompleta con datos          : q105 47.4%
Duplicados (filas completas)             : 0
record es identificador único            : True

Valores distintos en las columnas de texto/alta cardinalidad:
q5         35
q6orig     65
q7orig    356

Nulos del subconjunto (%):
redes_sociales_cod      24.4
salud_mental_cod        21.9
sueno_cod               13.2
actividad_fisica_cod     6.1
raceeth_cod              1.8
sexo_cod                 0.8
edad_cod                 0.5
estrato                  0.0
peso_muestral            0.0
id_registro              0.0
psu                      0.0


## 6. Control de versiones

**Qué hace este paso.** Consulta el estado de Git desde el propio cuaderno: instalación,
identidad configurada, rama, *commits*, cambios pendientes, remoto y autoría.

**Por qué se consulta y no se ejecuta.** Este cuaderno **no** hace `git add` ni `commit`:
versionar es una decisión de quien trabaja. Lo que sí corresponde es dejar constancia
verificable del estado del repositorio en el momento de la entrega.

In [16]:
def git(*argumentos):
    '''
    Ejecuta un comando de Git en la raíz del repositorio.

    Retorna
    -------
    (bool, str)
        Éxito de la ejecución y salida (o mensaje de error).
    '''
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], capture_output=True, text=True,
                                 encoding="utf-8", errors="replace", timeout=20, cwd=RAIZ)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f"No se pudo ejecutar Git: {e}"
    if proceso.returncode != 0:
        return False, proceso.stderr.strip() or "Git terminó con error."
    return True, proceso.stdout.rstrip()


ok, salida = git("--version")
print("Git                :", salida if ok else f"[AVISO] {salida}")
for clave in ("user.name", "user.email"):
    ok_cfg, valor = git("config", "--get", clave)
    print(f"{clave:19}:", valor if ok_cfg and valor else "[FALTA] configúrelo con git config --global")

ok_rama, rama = git("rev-parse", "--abbrev-ref", "HEAD")
print("Rama actual        :", rama if ok_rama else "sin commits todavía")
ok_n, n_commits = git("rev-list", "--count", "HEAD")
print("Commits en la rama :", n_commits if ok_n else "—")
ok_ramas, ramas = git("branch", "-r")
print("Ramas remotas      :", ", ".join(r.strip() for r in ramas.splitlines() if "->" not in r) if ok_ramas else "—")
ok_rem, remoto = git("remote", "get-url", "origin")
print("Remoto             :", remoto if ok_rem else "[FALTA] git remote add origin URL")

ok_log, log = git("log", "--oneline", "-8")
print("\nÚltimos commits\n" + (log if ok_log else "  (sin commits)"))

ok_st, estado = git("status", "--porcelain")
pendientes = [l for l in estado.splitlines() if l.strip()] if ok_st else []
print(f"\nArchivos con cambios sin registrar: {len(pendientes)}")
for linea in pendientes[:10]:
    print("  ", linea)

Git                : git version 2.55.0.windows.5
user.name          : DanielRamirezPerez
user.email         : d.ramrezprez@uandresbello.edu
Rama actual        : main
Commits en la rama : 14
Ramas remotas      : origin/main
Remoto             : https://github.com/MatiasManriquezO/proyecto-grupo8-mcdi500.git

Últimos commits
1feef0f docs: actualiza S1_F1_Definicion
a8b9505 fix: Se modifica ruta de exportación de F2
c35d574 fix: corrige .gitignore
b5054a8 fix: se elimina gitignore.txt
7716529 docs: se agrega .gitignore
587116f docs: corrige descripcion de q80 en procesamiento.py
9cb21b7 Merge branch 'main' of https://github.com/MatiasManriquezO/proyecto-grupo8-mcdi500
3de97bc docs: agrega diccionario, registro y metadatos de F1 y F2

Archivos con cambios sin registrar: 0


### 6.1 Autoría: un integrante, una identidad

La contribución individual se evalúa por el historial, así que cada integrante debe aparecer con
**su propia identidad**, y siempre la misma: el correo de su cuenta universitaria de GitHub. La
celda cuenta los *commits* por autor y avisa si algún integrante todavía no tiene *commits* propios.

> **Sobre este repositorio.** El grupo lo creó de nuevo para que los cuatro integrantes figuren con
> su cuenta universitaria (en el repositorio anterior había *commits* hechos con cuentas Gmail). El
> historial anterior se conserva en el repositorio original, `MatiasIMO98/proyecto-grupo8-mcdi500`.

In [17]:
def autoria():
    '''Cuenta commits por autor (nombre y correo con que quedaron registrados).'''
    ok, salida = git("log", "--format=%aN <%aE>")
    if not ok or not salida:
        return pd.Series(dtype=int)
    return pd.Series(salida.splitlines()).value_counts()


autores = autoria()
integrantes = len(PROYECTO["integrantes"])
no_universitarios = [a for a in autores.index if "uandresbello" not in a]
print(f"Identidades con commits : {len(autores)}")
print(f"Integrantes del grupo   : {integrantes}")
if len(autores) < integrantes:
    print(f"[AVISO] Faltan commits propios de {integrantes - len(autores)} integrante(s)")
if no_universitarios:
    print(f"[AVISO] Identidades sin correo universitario: {no_universitarios}")
print()
autores.rename("commits").to_frame()

Identidades con commits : 4
Integrantes del grupo   : 4



,commits
DanielRamirezPerez <d.ramrezprez@uandresbello.edu>,7
MatiasManriquezO <m.manriquezortiz@uandresbello.edu>,3
RobertSanchezS <r.snchezsaldivia@uandresbello.edu>,3
d.ramrezprez <d.ramrezprez@uandresbello.edu>,1


**Convención de *commits* del grupo.** Los prefijos `docs`, `data`, `feat` y `fix` permiten leer el
historial como una bitácora. La tabla no se escribe a mano: toma del historial el último *commit*
real de cada prefijo y cuenta cuántos mensajes no siguen la convención (por ejemplo, los que quedaron
con el mensaje por defecto de la interfaz web).

In [18]:
PREFIJOS = {"docs": "Documentación, README, informe", "data": "Incorporación o actualización de datos",
            "feat": "Código nuevo que aporta funcionalidad", "fix": "Corrección de un defecto"}

ok_msj, mensajes = git("log", "--no-merges", "--format=%h|%s")
filas = [linea.split("|", 1) for linea in mensajes.splitlines()] if ok_msj else []
convencion = []
for prefijo, cuando in PREFIJOS.items():
    ejemplos = [(h, m) for h, m in filas if m.lower().startswith(prefijo + ":")]
    convencion.append({"prefijo": prefijo, "cuándo": cuando, "commits": len(ejemplos),
                       "último ejemplo real": f"{ejemplos[0][0]} {ejemplos[0][1]}" if ejemplos else "—"})
fuera = [m for _, m in filas if not any(m.lower().startswith(p + ":") for p in PREFIJOS)]
print(f"Commits sin merge: {len(filas)} · fuera de la convención: {len(fuera)}")
for m in fuera[:5]:
    print("   ·", m)
pd.DataFrame(convencion)

Commits sin merge: 13 · fuera de la convención: 0


,prefijo,cuándo,commits,último ejemplo real
0,docs,"Documentación, README, informe",6,1feef0f docs: actualiza S1_F1_Definicion
1,data,Incorporación o actualización de datos,2,e370ee9 data: agrega dataset procesado por F2
2,feat,Código nuevo que aporta funcionalidad,0,—
3,fix,Corrección de un defecto,5,a8b9505 fix: Se modifica ruta de exportación d...


> **Atención.** Evite tildes en los mensajes de *commit*: según la terminal, aparecen corruptos en
> el historial compartido.

## 7. Validación técnica de la fase

La función `validar_fase1` reúne las comprobaciones de la fase con `assert`: si una condición no
se cumple, el cuaderno se detiene y deja constancia de la falla.

1. **Estructura** — existen las carpetas declaradas en el mapa conceptual.
2. **Artefactos** — `.gitignore`, `requirements.txt` (no vacío) y `README.md` presentes.
3. **Definición** — problema, pregunta, objetivos, supuestos y restricciones declarados.
4. **Coherencia** — el diccionario coincide con el módulo y la ficha suma 117 columnas.
5. **Módulo** — `src/procesamiento.py` expone las siete funciones del pipeline.

In [19]:
FUNCIONES_PIPELINE = ["cargar_datos", "seleccionar_columnas", "diagnosticar_datos",
                      "clasificar_variables", "transformar_datos", "validar_datos", "guardar_dataset"]


def validar_fase1(raiz, config, ficha, diccionario):
    '''
    Comprueba que la Fase 1 dejó lo que exige la rúbrica.
    Lanza AssertionError si alguna comprobación falla.
    '''
    print("VALIDACIÓN DE LA FASE 1")
    print("-" * 50)

    faltan = [p for p in ESTRUCTURA if not p.is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(ESTRUCTURA)} carpetas)")

    for nombre in (".gitignore", "requirements.txt", "README.md"):
        archivo = raiz / nombre
        assert archivo.exists() and archivo.stat().st_size > 0, f"{nombre} falta o está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")
    readme = (raiz / "README.md").read_text(encoding="utf-8")
    if config["repositorio"] not in readme:
        print("[AVISO] El README no incluye el enlace al repositorio")

    for clave in ("titulo", "problematica", "pregunta_principal", "objetivo_general"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    assert config["supuestos"] and config["restricciones"], "Faltan supuestos o restricciones."
    print("[OK] Definición del problema completa (supuestos y restricciones por separado)")

    assert len(diccionario) == len(procesamiento.COLUMNAS_ANALISIS) == 11
    assert sum(ficha["roles"].values()) == ficha["columnas"]
    print(f"[OK] Diccionario coherente con el módulo ({len(diccionario)} variables) "
          f"y ficha con {ficha['columnas']} columnas clasificadas")

    ausentes = [f for f in FUNCIONES_PIPELINE if not callable(getattr(procesamiento, f, None))]
    assert not ausentes, f"El módulo no expone: {ausentes}"
    print(f"[OK] src/procesamiento.py expone las {len(FUNCIONES_PIPELINE)} funciones del pipeline")

    print(f"\n[INFO] Conjunto declarado: {ficha['titulo']}")
    print(f"[INFO] Archivo descargado: {'sí' if ARCHIVO.exists() else 'todavía no'}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario)

VALIDACIÓN DE LA FASE 1
--------------------------------------------------
[OK] Estructura completa (9 carpetas)
[OK] .gitignore presente (1712 bytes)
[OK] requirements.txt presente (1928 bytes)
[OK] README.md presente (10373 bytes)
[AVISO] El README no incluye el enlace al repositorio
[OK] Definición del problema completa (supuestos y restricciones por separado)
[OK] Diccionario coherente con el módulo (11 variables) y ficha con 117 columnas clasificadas
[OK] src/procesamiento.py expone las 7 funciones del pipeline

[INFO] Conjunto declarado: 2023 National Youth Risk Behavior Survey (YRBS)
[INFO] Archivo descargado: sí


True

## 8. Vinculación con el mapa conceptual

La tabla hace explícita la correspondencia entre cada bloque del mapa conceptual (v5), dónde se
implementa y qué archivo lo respalda. Distingue además dos propiedades que no son lo mismo:
la **reproducibilidad** es una propiedad del **entorno** (`requirements.txt`, entorno aislado) y
la **trazabilidad** es una propiedad del **registro** (commits, bitácora). Un proyecto puede ser
reproducible y no trazable. La celda verifica que cada archivo de evidencia exista.

In [20]:
VINCULACION = [
    # (bloque del mapa, dónde se implementa, archivo de evidencia, propiedad, estado)
    ("1. Problema y preguntas", "Sección 1 · PROYECTO", "F1/notebooks/S1_F1_Definicion.ipynb", "definición", "implementado"),
    ("2. Datos: YRBS 2023", "Sección 5 · ficha y diccionario", "F1/data/raw/XXH2023_YRBSS_data.csv", "trazabilidad", "implementado"),
    ("2. Datos: diagnóstico y limpieza", "F2 · S1_F2_Preprocesamiento", "F2/notebooks/S1_F2_Preprocesamiento.ipynb", "trazabilidad", "implementado"),
    ("2. Datos: dataset procesado", "F2 · guardar_dataset", "F2/data/processed/yrbs2023_seleccion_procesada.csv", "reproducibilidad", "implementado"),
    ("3. Entorno y análisis", "Sección 2 · verificar_entorno", "requirements.txt", "reproducibilidad", "implementado"),
    ("3. Librerías y dependencias", "Sección 3.2 · verificar_requirements", "requirements.txt", "reproducibilidad", "implementado"),
    ("4. Control de versiones (Git)", "Sección 6 · git()", ".gitignore", "trazabilidad", "implementado"),
    ("5. GitHub (colaboración)", "Sección 6 · remoto, ramas y autoría", "README.md", "trazabilidad", "implementado"),
    ("6. Documentación científica", "Secciones 9-10 · metadatos y reflexión", "docs/bitacora_decisiones.md", "trazabilidad", "implementado"),
    ("7. Estructura del repositorio", "Sección 3 · árbol verificado", "README.md", "reproducibilidad", "implementado"),
    ("8. Commits reales", "Sección 6 · git log", "README.md", "trazabilidad", "implementado"),
    ("Análisis ponderado y modelación", "Fase 3 · núcleo algorítmico y POO", "F3/notebooks/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb", "—", "en desarrollo"),
    ("Visualización y comunicación", "Fase 4", "F4/notebooks/Fase 4.md", "—", "proyectado"),
]

vinculacion = pd.DataFrame(VINCULACION, columns=["bloque_mapa", "donde_se_implementa",
                                                 "evidencia", "propiedad", "estado"])
vinculacion["evidencia_existe"] = vinculacion["evidencia"].map(lambda r: (RAIZ / r).exists())

sin_evidencia = vinculacion.loc[~vinculacion["evidencia_existe"], "evidencia"].tolist()
if sin_evidencia:
    print("[AVISO] Evidencia declarada que no existe en el repositorio:", sin_evidencia)
else:
    print(f"[OK] {len(vinculacion)} filas de vinculación, todas con un archivo de evidencia existente.")
print(vinculacion["estado"].value_counts().to_string(), "\n")
vinculacion

[OK] 13 filas de vinculación, todas con un archivo de evidencia existente.
estado
implementado     11
en desarrollo     1
proyectado        1 



,bloque_mapa,donde_se_implementa,evidencia,propiedad,estado,evidencia_existe
0,1. Problema y preguntas,Sección 1 · PROYECTO,F1/notebooks/S1_F1_Definicion.ipynb,definición,implementado,True
1,2. Datos: YRBS 2023,Sección 5 · ficha y diccionario,F1/data/raw/XXH2023_YRBSS_data.csv,trazabilidad,implementado,True
2,2. Datos: diagnóstico y limpieza,F2 · S1_F2_Preprocesamiento,F2/notebooks/S1_F2_Preprocesamiento.ipynb,trazabilidad,implementado,True
3,2. Datos: dataset procesado,F2 · guardar_dataset,F2/data/processed/yrbs2023_seleccion_procesada...,reproducibilidad,implementado,True
4,3. Entorno y análisis,Sección 2 · verificar_entorno,requirements.txt,reproducibilidad,implementado,True
5,3. Librerías y dependencias,Sección 3.2 · verificar_requirements,requirements.txt,reproducibilidad,implementado,True
6,4. Control de versiones (Git),Sección 6 · git(),.gitignore,trazabilidad,implementado,True
7,5. GitHub (colaboración),"Sección 6 · remoto, ramas y autoría",README.md,trazabilidad,implementado,True
8,6. Documentación científica,Secciones 9-10 · metadatos y reflexión,docs/bitacora_decisiones.md,trazabilidad,implementado,True
9,7. Estructura del repositorio,Sección 3 · árbol verificado,README.md,reproducibilidad,implementado,True


> **Cómo se escribe esto en el informe.** Cada fila debe poder señalarse en el repositorio: un
> archivo, una celda o un *commit*. Un bloque del mapa que no tiene dónde apuntar es un bloque que
> se dibujó pero no se construyó.

## 9. Persistencia y trazabilidad

El resultado de la fase queda guardado en `F1/data/docs/` como tablas y metadatos que se versionan y se
citan en el informe técnico. El `README.md` del grupo ya existe y es más completo que uno
generado automáticamente, por lo que **no se sobrescribe**: se verificó en la sección 7.

In [21]:
diccionario.to_csv(DIR_SALIDA_F1 / "diccionario_variables_f1.csv", index=False)
comparacion.to_csv(DIR_SALIDA_F1 / "evaluacion_criterios_dataset.csv", index=False)
vinculacion.to_csv(DIR_SALIDA_F1 / "vinculacion_mapa_conceptual.csv", index=False)

METADATOS = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "repositorio": PROYECTO["repositorio"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": SEMILLA,
    "python": ENTORNO["python"],
    "sistema": f"{platform.system()} {platform.release()}",
    "tipo_entorno": ENTORNO["tipo_entorno"],
    "dependencias": ENTORNO["versiones"],
    "commit": git("rev-parse", "--short", "HEAD")[1],
    "dataset": {"titulo": FICHA["titulo"], "url": FICHA["url"],
                "archivo": ARCHIVO.relative_to(RAIZ).as_posix(),
                "archivo_descargado": ARCHIVO.exists(),
                "dimensiones_originales": [FICHA["filas"], FICHA["columnas"]],
                "dimensiones_subconjunto": [FICHA["filas"], len(procesamiento.COLUMNAS_ANALISIS)]},
    "criterios_incumplidos": incumplidos,
}
(DIR_SALIDA_F1 / "metadatos_fase1.json").write_text(
    json.dumps(METADATOS, indent=2, ensure_ascii=False), encoding="utf-8")

for nombre in ("diccionario_variables_f1.csv", "evaluacion_criterios_dataset.csv",
               "vinculacion_mapa_conceptual.csv", "metadatos_fase1.json"):
    archivo = DIR_SALIDA_F1 / nombre
    print(f"  {DIR_SALIDA_F1.relative_to(RAIZ).as_posix()}/{nombre:38} {archivo.stat().st_size:6} bytes")

  F1/data/docs/diccionario_variables_f1.csv             1270 bytes
  F1/data/docs/evaluacion_criterios_dataset.csv          333 bytes
  F1/data/docs/vinculacion_mapa_conceptual.csv          1573 bytes
  F1/data/docs/metadatos_fase1.json                      940 bytes


In [22]:
# Resumen de cierre de la fase.
resumen = pd.DataFrame([
    {"artefacto": "Carpetas verificadas", "estado": len(ESTRUCTURA)},
    {"artefacto": "Reglas activas en .gitignore", "estado": len(reglas_gitignore)},
    {"artefacto": "Dependencias del proyecto que coinciden con requirements.txt",
     "estado": f"{(tabla_requirements['coincide'] == 'sí').sum()} de {len(tabla_requirements)}"},
    {"artefacto": "Funciones del módulo probadas", "estado": 5},
    {"artefacto": "Columnas del archivo original clasificadas", "estado": FICHA["columnas"]},
    {"artefacto": "Variables en el diccionario del proyecto", "estado": len(diccionario)},
    {"artefacto": "Filas de vinculación con el mapa", "estado": len(vinculacion)},
    {"artefacto": "Criterios de dataset incumplidos (conjunto de trabajo)", "estado": len(incumplidos)},
])
resumen

,artefacto,estado
0,Carpetas verificadas,9
1,Reglas activas en .gitignore,31
2,Dependencias del proyecto que coinciden con re...,3 de 4
3,Funciones del módulo probadas,5
4,Columnas del archivo original clasificadas,117
5,Variables en el diccionario del proyecto,11
6,Filas de vinculación con el mapa,13
7,Criterios de dataset incumplidos (conjunto de ...,1


## Conclusiones y trazabilidad

La fase deja declarado el problema (pregunta, objetivos, alcance, supuestos y restricciones por
separado), un entorno verificado con evidencia, los artefactos de reproducibilidad comprobados
(`.gitignore`, `requirements.txt` no vacío, `README.md`), el módulo `src/procesamiento.py` probado en
casos normales, límite y de excepción, y la ficha del YRBS 2023 evaluada contra los criterios del
curso. Cada paso se implementó como función documentada y verificada (`presentar_proyecto`,
`presentar_delimitacion`, `encontrar_raiz`, `verificar_entorno`, `verificar_gitignore`,
`verificar_requirements`, `contar_roles`, `evaluar_criterios`, `git`, `autoria`, `validar_fase1`).

**Trazabilidad con el repositorio:** este cuaderno está en `F1/notebooks/`; el dato original en
`F1/data/raw/`; las decisiones con su cifra en `docs/bitacora_decisiones.md`; las tablas de esta
fase en `docs/`, y el historial de *commits* registra el avance descrito.

---

## 10. Reflexión técnica

### Hallazgos

- **El primer conjunto no podía responder la pregunta.** El dataset de Kaggle tenía categorías
  repartidas en proporciones exactas y coeficientes ≈ 0 entre todas las variables. Se reemplazó por
  el YRBS 2023 antes de construir el pipeline (bitácora 0.1–0.3).
- **El YRBS cumple los criterios del curso salvo la fecha**, y ese incumplimiento viene del diseño
  transversal de la encuesta, no de la selección. El texto de alta cardinalidad sí existe: `q5`
  tiene 35 combinaciones de respuesta múltiple.
- **Archivo original y subconjunto son cifras distintas:** 20.103 × 117 frente a 20.103 × 11. Se
  reducen columnas, no filas.
- **La escala de `q80` es de frecuencia, no de horas.** Va de *"no uso redes sociales"* a *"más de
  una vez por hora"*; cualquier interpretación en horas diarias sería incorrecta.
- **Las pruebas encontraron un comportamiento silencioso:** `seleccionar_columnas` omite una
  columna ausente sin avisar; solo `validar_datos` lo detecta. La validación no es opcional.
- **La reproducibilidad es verificable o no existe.** Un `requirements.txt` vacío no se nota al
  trabajar en el computador propio; solo aparece al clonar. Por eso esta fase lo verifica con código.

### Dificultades y cómo se resolvieron

- **Rutas entre cuadernos.** Se fijó una sola convención (`F1/data/raw/` para el original,
  `F2/data/processed/` para la salida) y el cuaderno comprueba que F2 la respete.
- **Identidades de Git.** En el primer repositorio los integrantes registraron *commits* con
  varias identidades, algunas con cuentas Gmail. Se creó un repositorio nuevo en el que cada
  integrante trabaja con su cuenta universitaria, y la sección 6.1 comprueba que así sea.

### Decisiones que quedan abiertas

- Cómo incorporar la ponderación (`weight`, `stratum`, `psu`) en el análisis de la Fase 3.
- Si `q76` (días de actividad física) se mantiene como ordinal o pasa a discreta.
- Cómo declarar en el informe la ausencia de variable de fecha frente al criterio del curso.
- Si parte de `src/procesamiento.py` pasa a clases en la Fase 3.

---

## 11. Verificación antes de entregar

**Cuaderno**
- Corre completo tras *Kernel → Restart Kernel and Run All Cells*, sin errores.
- La numeración de ejecución es continua.
- Cada bloque de código está precedido por una celda narrativa.
- Todas las rutas son relativas a la raíz del repositorio y la semilla está declarada.

**Contenido técnico**
- Problemática, pregunta, objetivos, alcance, supuestos y restricciones declarados por separado.
- El entorno se verifica con evidencia (intérprete, tipo de entorno, versiones).
- La procedencia del conjunto está documentada, con enlace y cita en APA 7.
- El diccionario declara el rol analítico de cada variable y coincide con el módulo.
- El conjunto fue evaluado contra los criterios del curso.
- Hay pruebas de casos normales, límite y excepciones.

**Archivos**
- `.gitignore` y `requirements.txt` en la raíz, ambos con contenido.
- `README.md` con enlace al repositorio.
- `docs/` con diccionario, evaluación del conjunto, vinculación y metadatos.

**Repositorio**
- Carpeta F1 con este cuaderno, *commits* de todos los integrantes y `README` actualizado.
- El correo de Git de cada integrante coincide con el de su cuenta de GitHub.

---

*Cuaderno de la Fase 1 · Grupo 8 · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*